# Notebook 11: LightGBM consumption huis
---
- LightGBM-model dat het verbruik (`consumption`) van het **huis** 3 dagen vooruit voorspelt, op 5-minutenresolutie.
- De split komt uit `06_Dataprep_v3` (stap 4.0): vaste validatie- en testvensters per seizoen, voor alle modellen hetzelfde. Alleen de train-data is ingevuld.
- Scoren gaat met de gedeelde functie uit `models/evaluatie.py`: rolling evaluatie met een vast model, op 5, 10, 15, 30 en 60 minuten en 1, 2 en 3 dagen vooruit.
- 7 dagen historie, 3 dagen horizon, elk uur een nieuwe forecast.
- Runtime van het model (features bouwen, trainen, voorspellen) wordt bijgehouden met `time`.

### Inladen
---
1. Libraries en paths inladen

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import lightgbm as lgb

start_notebook = time.perf_counter()
runtimes = {}                                   # hierin komen alle gemeten tijden (seconden)

#Projectmap zoeken, werkt vanuit elke submap van models/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "prepared_data").exists())
sys.path.append(str(ROOT / "models"))

import evaluatie as ev                          # gedeelde vensters en scorefunctie voor alle modellen

DATA_BESTAND = "forecast_targets_with_weather.parquet"
print("LightGBM versie:", lgb.__version__)

LightGBM versie: 4.7.0


2. Data inladen
---
- `ev.laad_reeks` geeft de reeks als volledig 5-minutenraster, met de kolom `set` (train, val, test of stresstest).
- Er worden geen rijen meer weggegooid. Anders kloppen de rijnummers niet meer met de tijd en liggen de vensters niet voor elk model op dezelfde plek. LightGBM kan zelf met lege waarden omgaan.
- Het bestand met weerdata moet de kolom `set` hebben: draai eerst `06_Dataprep_v3` en daarna `08_Weermerge`.

In [2]:
t0 = time.perf_counter()
huis_cons = ev.laad_reeks("huis", "consumption", bestand=DATA_BESTAND)
runtimes["data inladen"] = time.perf_counter() - t0

print(huis_cons.shape)
print("Periode:", huis_cons["timestamp"].min(), "t/m", huis_cons["timestamp"].max())
huis_cons["set"].value_counts()

ValueError: 'forecast_targets_with_weather.parquet' heeft geen kolom 'set'. Draai eerst 06_Dataprep_v3

### Voorspel instellingen
---
- LightGBM is geen sequentiemodel. Daarom wordt de forecast als tabel opgebouwd: elke rij is één combinatie van **forecastmoment** (origin) en **stap vooruit** (1 t/m 864).
- De horizon-stap is zelf een feature, zodat één model alle 864 stappen voorspelt.

In [ ]:
stap_per_dag = ev.PUNTEN_PER_DAG            # 288 stappen per dag (5 minuten)

aantal_dagen_input = 7                     # historie die beschikbaar moet zijn
input_len = aantal_dagen_input * stap_per_dag

aantal_dagen_forecast = 3                  # horizon
horizon = aantal_dagen_forecast * stap_per_dag

stride_eval = 12                           # validatie en test: elk uur een nieuwe forecast (zelfde als de andere modellen)
stride_train = 72                          # trainvensters elke 6 uur (lager = meer data, meer geheugen/tijd)

# In train is dit de meting met ingevulde gaten, in validatie en test de ruwe meting (niets ingevuld)
TARGET_COL = "pow_imputed"

# Weer op het voorspelde tijdstip gebruiken? True = aanname dat er een (perfecte) weersverwachting is.
# False = alleen weer tot aan het forecastmoment (eerlijker vergelijk met het GRU-model).
# Bij verbruik is het effect van weer kleiner dan bij opwek (zie correlaties in het GRU-notebook).
GEBRUIK_TOEKOMST_WEER = True

### Features
---
1. Features per tijdstip (kalender + weer)

In [ ]:
tijd = pd.to_datetime(huis_cons["timestamp"])
minuut_van_dag = tijd.dt.hour * 60 + tijd.dt.minute
dag_van_week = tijd.dt.dayofweek + tijd.dt.hour / 24
dag_van_jaar = tijd.dt.dayofyear

tijd_features = pd.DataFrame({
    "uur_sin":  np.sin(2 * np.pi * minuut_van_dag / 1440),
    "uur_cos":  np.cos(2 * np.pi * minuut_van_dag / 1440),
    "week_sin": np.sin(2 * np.pi * dag_van_week / 7),
    "week_cos": np.cos(2 * np.pi * dag_van_week / 7),
    "jaar_sin": np.sin(2 * np.pi * dag_van_jaar / 365.25),
    "jaar_cos": np.cos(2 * np.pi * dag_van_jaar / 365.25),
    "dag_van_week": tijd.dt.dayofweek,
    "is_weekend": (tijd.dt.dayofweek >= 5).astype(int),
})

rad = np.deg2rad(huis_cons["Windrichting"])
weer_features = pd.DataFrame({
    "Temperatuur":      huis_cons["Temperatuur"],
    "GlobaleStraling":  huis_cons["GlobaleStraling"],
    "DuurNeerslag":     huis_cons["DuurNeerslag"],
    "SomNeerslag":      huis_cons["SomNeerslag"],
    "WindSnelheid":     huis_cons["WindSnelheid"],
    "LuchtVochtigheid": huis_cons["LuchtVochtigheid"],
    "wind_sin":         np.sin(rad),
    "wind_cos":         np.cos(rad),
})

TIJD_COLS = list(tijd_features.columns)
WEER_COLS = list(weer_features.columns)
tijd_arr = tijd_features.to_numpy(np.float32)
weer_arr = weer_features.to_numpy(np.float32)
y_arr = huis_cons[TARGET_COL].to_numpy(np.float64)

# In validatie en test zitten lege punten. Gemiddelden tellen daarom alleen de bekende waarden mee
bekend = ~np.isnan(y_arr)
cumsum = np.concatenate([[0.0], np.cumsum(np.nan_to_num(y_arr))])     # voor snelle rolling gemiddelden
cumaantal = np.concatenate([[0], np.cumsum(bekend)])
y_laatst_bekend = pd.Series(y_arr).ffill().to_numpy()                 # laatste bekende waarde, kijkt alleen terug
straling_24u_arr = (pd.Series(weer_features["GlobaleStraling"].to_numpy(np.float64))
                    .rolling(stap_per_dag, min_periods=1).mean().to_numpy(np.float32))

n = len(y_arr)
print(f"n = {n} | bekend: {bekend.mean() * 100:.1f}%")

2. Rijen bouwen per forecastmoment
---
Voor forecastmoment `o` (eerste voorspelde index) en stap `h` is het doel `y[o + h]`. Er wordt alleen data van vóór `o` gebruikt:
- `laatste`, `gem_1u`, `gem_24u`, `gem_7d`: niveau van het verbruik vlak voor het forecastmoment
- `zelfde_tijd_recent`: verbruik op hetzelfde tijdstip van de dag, de meest recente dag die al bekend is
- `zelfde_tijd_gem7`: gemiddelde daarvan over 7 dagen
- `zelfde_tijd_vorige_week`: verbruik exact één week eerder op hetzelfde tijdstip (weekpatroon; altijd bekend omdat de horizon < 7 dagen)
- `zelfde_tijd_vorige_week_gem1u`: gemiddeld verbruik rond dat tijdstip een week eerder (±30 min), minder ruis dan één 5-minutenwaarde
- `straling_24u`: gemiddelde straling over de laatste 24 uur
- `stap`, `dag_vooruit`: hoe ver vooruit
- kalender (en eventueel weer) op het voorspelde tijdstip

Lege punten in de historie blijven leeg (NaN) of vallen uit het gemiddelde. Er wordt in validatie en test niets ingevuld.

**Forecastmomenten**
- Train: `ev.train_origins`. De 7 dagen historie en de 3 dagen horizon liggen volledig in train, zodat er geen punt uit validatie of test in een trainvoorbeeld zit. Rijen zonder doelwaarde vallen af.
- Validatie en test: `ev.maak_origins`. Vaste momenten binnen de vensters, voor elk model hetzelfde.

In [ ]:
def gem_venster(eind, lengte):
    """Gemiddelde van de bekende waarden in y[eind-lengte : eind] (NaN als er geen bekend is)."""
    aantal = cumaantal[eind] - cumaantal[eind - lengte]
    som = cumsum[eind] - cumsum[eind - lengte]
    return np.where(aantal > 0, som / np.maximum(aantal, 1), np.nan)


def bouw_rijen(origins):
    origins = np.asarray(origins)
    h = np.tile(np.arange(horizon), len(origins))
    o = np.repeat(origins, horizon)
    t = o + h                                        # index van het voorspelde tijdstip
    k = h // stap_per_dag + 1                        # aantal dagen terug zodat het tijdstip al bekend is

    # zelfde tijdstip op de 7 meest recente bekende dagen, gemiddeld over de dagen met een waarde
    dagen7 = np.stack([y_arr[t - stap_per_dag * (k + j)] for j in range(7)])
    aantal7 = (~np.isnan(dagen7)).sum(axis=0)
    gem7 = np.where(aantal7 > 0, np.nansum(dagen7, axis=0) / np.maximum(aantal7, 1), np.nan)

    X = {
        "stap": h.astype(np.float32),
        "dag_vooruit": (k - 1).astype(np.float32),
        "laatste": y_laatst_bekend[o - 1],
        "gem_1u": gem_venster(o, 12),
        "gem_24u": gem_venster(o, stap_per_dag),
        "gem_7d": gem_venster(o, input_len),
        "zelfde_tijd_recent": dagen7[0],
        "zelfde_tijd_gem7": gem7,
        "zelfde_tijd_vorige_week": y_arr[t - input_len],
        "zelfde_tijd_vorige_week_gem1u": gem_venster(np.minimum(t - input_len + 6, o), 12),
        "straling_24u": straling_24u_arr[o - 1],
    }
    X = pd.DataFrame(X).astype(np.float32)
    X[TIJD_COLS] = tijd_arr[t]
    if GEBRUIK_TOEKOMST_WEER:
        X[WEER_COLS] = weer_arr[t]                   # weer op het voorspelde tijdstip
    else:
        X[WEER_COLS] = weer_arr[o - 1]               # laatst bekende weer
    return X, y_arr[t]


def bouw_trainset(origins):
    """Rijen bouwen en de rijen zonder doelwaarde laten vallen."""
    X, y = bouw_rijen(origins)
    heeft_doel = ~np.isnan(y)
    return X[heeft_doel].reset_index(drop=True), y[heeft_doel]


# train: historie en horizon volledig in train
train_origins = ev.train_origins(huis_cons, input_len=input_len, horizon=horizon, stap=stride_train, volledig=False)
# validatie en test: vaste momenten binnen de vensters, voor elk model hetzelfde
val_origins = ev.maak_origins(huis_cons, "val", stap=stride_eval, horizon=horizon)
test_origins = ev.maak_origins(huis_cons, "test", stap=stride_eval, horizon=horizon)

t0 = time.perf_counter()
X_train, y_train = bouw_trainset(train_origins)
X_val, y_val = bouw_trainset(val_origins)
runtimes["features bouwen"] = time.perf_counter() - t0

FEATURES = list(X_train.columns)
print(f"forecastmomenten | train: {len(train_origins)} | val: {len(val_origins)} | test: {len(test_origins)}")
print(f"rijen            | train: {len(X_train):,} | val: {len(X_val):,}")
print(f"aantal features: {len(FEATURES)}")
print(f"Features bouwen: {runtimes['features bouwen']:.1f}s")

### Model
---
- Objective `l1` (MAE), net als de L1-loss van de GRU.
- `dag_van_week` wordt als categorische feature meegegeven.
- Early stopping op de validatievensters. Na het trainen blijft het model vast.

In [ ]:
params = {
    "objective": "l1",
    "metric": "l1",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}
n_rondes = 3000

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=["dag_van_week"], free_raw_data=False)
dval = lgb.Dataset(X_val, label=y_val, categorical_feature=["dag_van_week"], reference=dtrain, free_raw_data=False)

evals = {}
start_totaal = time.perf_counter()                  # start tijd
model = lgb.train(
    params,
    dtrain,
    num_boost_round=n_rondes,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100), lgb.record_evaluation(evals)],
)
runtimes["training"] = time.perf_counter() - start_totaal

print(f"\nTotale trainingstijd: {runtimes['training']:.1f}s ({runtimes['training'] / 60:.1f} min)")
print(f"Beste iteratie: {model.best_iteration} | gemiddeld per boosting-ronde: "
      f"{runtimes['training'] / len(evals['train']['l1']) * 1000:.1f} ms")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(evals["train"]["l1"], label="Train")
plt.plot(evals["val"]["l1"], label="Validatie")
plt.axvline(model.best_iteration, color="black", linestyle="--", alpha=0.5, label="Beste iteratie")
plt.xlabel("Boosting-ronde")
plt.ylabel("MAE (W)")
plt.title("Loss per boosting-ronde")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
importance = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values()
plt.figure(figsize=(8, 6))
(importance / importance.sum() * 100).plot.barh()
plt.xlabel("Aandeel gain (%)")
plt.title("Feature importance")
plt.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### Rolling evaluatie op validatie
---
- `voorspel(origins)` geeft per forecastmoment de 864 voorspelde stappen terug. Meer hoeft een notebook niet te leveren.
- `ev.rolling_evaluatie` loopt de forecastmomenten op volgorde van tijd af. Het model blijft hetzelfde en krijgt steeds iets meer data.
- Gescoord wordt op 5, 10, 15, 30 en 60 minuten en 1, 2 en 3 dagen vooruit, alleen tegen echte metingen (`pow_w`).
- Ter vergelijking: *zelfde tijd gisteren* (`ev.seasonal_naive`) op precies dezelfde momenten.

In [ ]:
def voorspel(origins, blok=500):
    preds = []
    for i in range(0, len(origins), blok):
        X_blok, _ = bouw_rijen(origins[i:i + blok])
        preds.append(model.predict(X_blok, num_iteration=model.best_iteration).reshape(-1, horizon))
    return np.vstack(preds)


voorspel_gisteren = ev.seasonal_naive(huis_cons)


def vergelijk(detail, detail_gisteren):
    """MAE en RMSE per horizon van LightGBM naast 'zelfde tijd gisteren'."""
    eigen, basis = ev.samenvatting(detail), ev.samenvatting(detail_gisteren)
    return pd.DataFrame({"aantal": eigen["aantal"],
                         "MAE LightGBM": eigen["MAE"],
                         "MAE zelfde tijd gisteren": basis["MAE"],
                         "RMSE LightGBM": eigen["RMSE"],
                         "RMSE zelfde tijd gisteren": basis["RMSE"]}).reindex(list(ev.HORIZONS)).round(1)


detail_val = ev.rolling_evaluatie(voorspel, huis_cons, val_origins)
detail_val_gisteren = ev.rolling_evaluatie(voorspel_gisteren, huis_cons, val_origins)

print(f"Validatie: {len(val_origins)} forecastmomenten")
vergelijk(detail_val, detail_val_gisteren)

- Dezelfde MAE per seizoen. Zo is te zien of het model in één seizoen duidelijk slechter is.

In [ ]:
ev.samenvatting(detail_val, per_venster=True)["MAE"].unstack("venster").reindex(list(ev.HORIZONS)).round(1)

### Evaluatie op de testset
---
- Pas bekijken als het model en de instellingen vaststaan: de test is alleen voor de eindscore.
- Zelfde functie en zelfde horizons als bij validatie, nu op de vier testvensters.

In [ ]:
t0 = time.perf_counter()
pred_test = voorspel(test_origins)
runtimes["voorspellen test"] = time.perf_counter() - t0
print(f"Voorspellen {len(test_origins)} forecasts: {runtimes['voorspellen test']:.1f}s "
      f"({runtimes['voorspellen test'] / len(test_origins) * 1000:.1f} ms per forecast van {horizon} stappen)")

# de voorspellingen zijn al gemaakt, de functie hoeft ze alleen nog op te zoeken
positie = {o: i for i, o in enumerate(test_origins)}
detail_test = ev.rolling_evaluatie(lambda origins: pred_test[[positie[o] for o in origins]], huis_cons, test_origins)
detail_test_gisteren = ev.rolling_evaluatie(voorspel_gisteren, huis_cons, test_origins)

vergelijk(detail_test, detail_test_gisteren)

- MAE per voorspelstap over de hele horizon, en twee voorbeelden van een forecast. Lege stukken in de lijn "Echt" zijn tijdstippen zonder meting.

In [ ]:
# echte metingen (niet ingevuld) op de voorspelde tijdstippen
h = np.arange(horizon)
t_test = test_origins[:, None] + h[None, :]
echt_test = huis_cons["pow_w"].to_numpy(np.float64)[t_test]
gisteren_test = voorspel_gisteren(test_origins)

mae_per_stap = np.nanmean(np.abs(pred_test - echt_test), axis=0)
mae_gisteren_per_stap = np.nanmean(np.abs(gisteren_test - echt_test), axis=0)

plt.figure(figsize=(10, 3.5))
plt.plot(h / stap_per_dag, mae_per_stap, label="LightGBM")
plt.plot(h / stap_per_dag, mae_gisteren_per_stap, alpha=0.6, label="Zelfde tijd gisteren")
plt.xlabel("Dagen vooruit")
plt.ylabel("MAE (W)")
plt.title("MAE per voorspelstap (test)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


def plot_voorspelling(idx=0, titel="Voorspelling vs. echt"):
    o = test_origins[idx]
    hist = y_arr[o - input_len:o]
    echt, pred = echt_test[idx], pred_test[idx]
    mae = np.nanmean(np.abs(pred - echt))

    t_hist = np.arange(-input_len, 0) / stap_per_dag     # x-as in dagen
    t_fc = np.arange(horizon) / stap_per_dag

    plt.figure(figsize=(12, 4))
    plt.plot(t_hist, hist, color="grey", label="Input (historie)")
    plt.plot(t_fc, echt, label="Echt")
    plt.plot(t_fc, pred, label="Voorspelling")
    plt.axvline(0, color="black", linestyle="--", alpha=0.5)
    plt.xlabel("Dagen t.o.v. start voorspelling")
    plt.ylabel("Verbruik (W)")
    plt.title(f"{titel} (MAE: {mae:.0f} W)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

plot_voorspelling(idx=0)
plot_voorspelling(idx=len(test_origins) // 2)   # een ander moment

### Runtime overzicht
---

In [ ]:
runtimes["totaal notebook"] = time.perf_counter() - start_notebook
runtime_df = pd.DataFrame({"seconden": runtimes}).assign(minuten=lambda d: d["seconden"] / 60).round(2)
runtime_df

### Resultaten opslaan
---
- Rolling uitkomsten (validatie en test) gaan via `ev.bewaar` naar `models/results/rolling/`. Het vergelijkingsnotebook leest ze daar in met `ev.laad_resultaten()`.
- Het oude overzicht in `models/results/` blijft bestaan in hetzelfde format, nu berekend op de vier testvensters en alleen tegen echte metingen.
- Persistence dag = zelfde tijdstip van de meest recente bekende dag, persistence week = zelfde tijdstip een week eerder

In [ ]:
RESULTS_DIR = ROOT / "models" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAAM = "11_lightgbm"
print(ev.bewaar(detail_val, MODEL_NAAM, "huis", "consumption"))
print(ev.bewaar(detail_test, MODEL_NAAM, "huis", "consumption"))

def scores(pred, echt):
    fout = pred - echt                               # NaN waar geen echte meting is
    res = {"MAE": np.nanmean(np.abs(fout)), "RMSE": np.sqrt(np.nanmean(fout ** 2))}
    for d in range(aantal_dagen_forecast):
        s = slice(d * stap_per_dag, (d + 1) * stap_per_dag)
        res[f"MAE dag {d+1}"] = np.nanmean(np.abs(fout[:, s]))
    #Rolling validation: elke dag een nieuwe forecast
    dagelijks = slice(None, None, stap_per_dag // stride_eval)
    res["rolling MAE"] = np.nanmean(np.nanmean(np.abs(fout[dagelijks]), axis=1))
    res["rolling RMSE"] = np.nanmean(np.sqrt(np.nanmean(fout[dagelijks] ** 2, axis=1)))
    return res

y_gevuld = ev.vul_input(huis_cons["pow_w"].to_numpy(np.float64))     # kijkt alleen terug
base_week = y_gevuld[t_test - input_len]

alle = {"LightGBM": pred_test, "Persistence dag": gisteren_test, "Persistence week": base_week}
resultaten = pd.DataFrame({naam: scores(pred, echt_test) for naam, pred in alle.items()}).T
resultaten.loc["LightGBM", "trainingstijd (s)"] = runtimes["training"]
resultaten.loc["LightGBM", "voorspeltijd (s)"] = runtimes["voorspellen test"]
resultaten = resultaten.round(1)

#Model opslaan en bestandsgrootte meten
MODEL_DIR = ROOT / "models" / "saved_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def grootte_mb(pad):
    pad = Path(pad)
    if pad.is_dir():
        return sum(f.stat().st_size for f in pad.rglob("*") if f.is_file()) / 1e6
    return pad.stat().st_size / 1e6

pad = MODEL_DIR / "11_lightgbm_huis_cons.txt"
model.save_model(str(pad), num_iteration=model.best_iteration)
resultaten.loc["LightGBM", "modelgrootte (MB)"] = grootte_mb(pad)
resultaten["modelgrootte (MB)"] = resultaten["modelgrootte (MB)"].round(3)

KOLOMMEN = ["notebook", "locatie", "target", "resolutie", "testperiode", "MAE", "RMSE", "MAE dag 1", "MAE dag 2",
            "MAE dag 3", "rolling MAE", "rolling RMSE", "trainingstijd (s)", "voorspeltijd (s)",
            "modelgrootte (MB)"]
test_tijden = huis_cons.loc[huis_cons["set"] == "test", "timestamp"]
resultaten["notebook"] = "11_LightGBM_huis_cons"
resultaten["locatie"] = "huis"
resultaten["target"] = "consumption"
resultaten["resolutie"] = "5 min"
resultaten["testperiode"] = f"4 seizoensvensters, {test_tijden.min():%Y-%m-%d} t/m {test_tijden.max():%Y-%m-%d}"
resultaten = resultaten.reindex(columns=KOLOMMEN)
resultaten.to_csv(RESULTS_DIR / "11_lightgbm_huis_cons.csv")
resultaten